In [ ]:
import streamlit as st
import pandas as pd
import numpy as np
import subprocess
import os
import time
import signal
import sys
import re  # [新增] 用於文字清洗
from datetime import datetime

# ==========================================
# 0. 系統環境檢查與匯入
# ==========================================
st.set_page_config(
    page_title="News Viral Lens | 全域戰情室",
    page_icon="🛡️",
    layout="wide",
    initial_sidebar_state="expanded"
)

# 匯入核心模組
try:
    from inference_engine import NewsPredictor
    from fact_checker import FactChecker
except ImportError as e:
    st.error(f"⚠️ 核心模組匯入失敗: {e}")
    st.stop()

# 設定常數
CRAWLER_SCRIPT = "News_Scraper_Optimized.py"
DATA_FILE = "News_Data.csv"

# ==========================================
# 1. 文字清洗模組 (Data Cleaning Pipeline)
# ==========================================
def clean_text(text):
    """
    [新增] 自動清洗爬蟲雜訊
    """
    if not isinstance(text, str):
        return ""
    
    # 1. 去除 HTML 殘渣 (簡單版)
    text = re.sub(r'<[^>]+>', '', text)
    
    # 2. 去除網址 (http/https)
    text = re.sub(r'http\S+', '', text)
    
    # 3. 去除 Email
    text = re.sub(r'\S+@\S+', '', text)
    
    # 4. 去除多餘空白與換行 (將 \n\n 變成單一空格)
    text = re.sub(r'\s+', ' ', text)
    
    # 5. 去除常見無意義結尾 (可依需求擴充)
    stop_phrases = ["點我訂閱", "延伸閱讀", "加入會員", "更多新聞請看"]
    for phrase in stop_phrases:
        text = text.replace(phrase, '')
        
    return text.strip()

# ==========================================
# 2. 介面樣式優化
# ==========================================
st.markdown("""
    <style>
    @import url('https://fonts.googleapis.com/css2?family=Noto+Sans+TC:wght@400;700&display=swap');
    
    html, body, [class*="css"] {
        font-family: 'Noto Sans TC', sans-serif;
    }
    
    .news-container {
        background-color: #FFFFFF;
        border: 1px solid #E2E8F0;
        border-radius: 12px;
        padding: 20px;
        margin-bottom: 15px;
        box-shadow: 0 2px 4px rgba(0,0,0,0.05);
        transition: transform 0.2s;
    }
    .news-container:hover {
        transform: translateY(-2px);
        box-shadow: 0 8px 16px rgba(0,0,0,0.1);
    }
    
    .risk-badge {
        display: inline-block;
        padding: 4px 12px;
        border-radius: 99px;
        font-weight: bold;
        font-size: 0.85rem;
        margin-bottom: 8px;
    }
    .risk-high { background-color: #FEE2E2; color: #DC2626; border: 1px solid #FECACA; }
    .risk-mid { background-color: #FEF3C7; color: #D97706; border: 1px solid #FDE68A; }
    .risk-low { background-color: #DCFCE7; color: #16A34A; border: 1px solid #BBF7D0; }
    </style>
""", unsafe_allow_html=True)

# ==========================================
# 3. 後台爬蟲控制核心
# ==========================================

def is_crawler_running():
    try:
        if os.name == 'posix':
            cmd = ["pgrep", "-f", CRAWLER_SCRIPT]
            result = subprocess.run(cmd, capture_output=True)
            return result.returncode == 0
        else:
            return st.session_state.get('crawler_active', False)
    except:
        return False

def start_crawler():
    if not is_crawler_running():
        subprocess.Popen([sys.executable, CRAWLER_SCRIPT])
        st.session_state['crawler_active'] = True
        st.toast("🚀 爬蟲核心已啟動！", icon="🤖")
        time.sleep(3)
    else:
        st.toast("⚠️ 爬蟲已經在背景執行中", icon="ℹ️")

def stop_crawler():
    if os.name == 'posix':
        os.system(f"pkill -f {CRAWLER_SCRIPT}")
        os.system("pkill -f chromedriver")
    else:
        os.system(f"taskkill /F /IM python.exe /FI \"WINDOWTITLE eq {CRAWLER_SCRIPT}\"")
        os.system("taskkill /F /IM chromedriver.exe")
    
    st.session_state['crawler_active'] = False
    st.toast("🛑 爬蟲已停止", icon="✅")

# ==========================================
# 4. 初始化 AI 引擎
# ==========================================
@st.cache_resource
def load_ai_resources():
    with st.spinner("正在載入 Stacking 模型..."):
        try:
            predictor = NewsPredictor()
            checker = FactChecker()
            return predictor, checker
        except Exception as e:
            st.error(f"模型載入失敗: {e}")
            return None, None

predictor, checker = load_ai_resources()

# ==========================================
# 5. 側邊欄與主畫面
# ==========================================
with st.sidebar:
    st.title("🛡️ 戰情控制台")
    st.markdown("---")
    
    if is_crawler_running():
        st.success("🟢 爬蟲運作中")
        if st.button("🛑 停止監控"):
            stop_crawler()
            st.rerun()
    else:
        st.error("🔴 爬蟲已停止")
        if st.button("🚀 啟動全自動監控"):
            start_crawler()
            st.rerun()

    st.markdown("---")
    auto_refresh = st.toggle("開啟自動刷新", value=True)
    refresh_rate = st.slider("刷新頻率 (秒)", 5, 60, 10)

st.title("News Viral Lens 全域戰情室")

# 讀取數據
if os.path.exists(DATA_FILE):
    try:
        df = pd.read_csv(DATA_FILE)
        
        if not df.empty:
            if 'fetch_time' in df.columns:
                df['fetch_time'] = pd.to_datetime(df['fetch_time'])
                df = df.sort_values(by='fetch_time', ascending=False)
            
            # 顯示統計
            m1, m2 = st.columns(2)
            m1.metric("監控新聞總量", f"{len(df)} 篇")
            m2.metric("最新進稿", df.iloc[0]['media'])
            st.markdown("---")

            # 顯示列表 (最新 5 筆)
            latest_news = df.head(5)
            
            for idx, row in latest_news.iterrows():
                # [關鍵] 在這裡進行清洗！
                raw_title = str(row['title'])
                raw_content = str(row['content'])
                
                # 清洗後的乾淨文本 (給 AI 用，也給 UI 顯示用)
                clean_title = clean_text(raw_title)
                clean_content = clean_text(raw_content)
                
                # AI 預測
                try:
                    # 這裡使用乾淨的文本進行預測
                    # (假設您已經將 inference_engine 接好，這裡先用 Mock)
                    import random
                    viral_score = random.uniform(0.1, 0.95)
                    if "網傳" in clean_title: viral_score = 0.95
                except:
                    viral_score = 0.5
                
                # 燈號邏輯
                if viral_score > 0.8:
                    risk_class, risk_label, emoji = "risk-high", "P0 紅色警戒", "🔥"
                elif viral_score > 0.5:
                    risk_class, risk_label, emoji = "risk-mid", "P1 黃色觀察", "⚠️"
                else:
                    risk_class, risk_label, emoji = "risk-low", "P2 綠色安全", "✅"

                # 外部查證 (用標題查證)
                fc_result = checker.check_coverage(clean_title)
                
                # UI 渲染
                with st.container():
                    st.markdown(f"""
                    <div class="news-container">
                        <div class="risk-badge {risk_class}">{emoji} {risk_label} | 熱度預測: {int(viral_score*100)}%</div>
                        <h3 style="margin: 5px 0 10px 0;">{clean_title}</h3>
                        <div style="font-size: 0.85rem; color: #64748B; margin-bottom: 10px;">
                            📅 {row.get('publish_time', 'Unknown')} | 📰 {row.get('media', 'Unknown')} | 🔗 <a href="{row.get('url', '#')}" target="_blank">原始連結</a>
                        </div>
                        <div style="font-size: 0.95rem; color: #334155; line-height: 1.5; margin-bottom: 15px;">
                            {clean_content[:150]}... 
                        </div>
                    """, unsafe_allow_html=True)
                    # (注意：這裡顯示的也是 clean_content，閱讀體驗會比較好)

                    c1, c2 = st.columns([3, 1])
                    with c1:
                        if fc_result['is_isolated']:
                            st.warning(f"⚠️ **孤證警訊**：{fc_result['status_text']}")
                        else:
                            st.success(f"✅ **多方證實**：{fc_result['status_text']}")
                    
                    st.markdown("</div>", unsafe_allow_html=True)

        else:
            st.info("📭 等待資料進稿中...")
            
    except Exception as e:
        st.error(f"讀取數據錯誤: {e}")
else:
    st.warning("⚠️ 請點擊左側啟動爬蟲")

if auto_refresh and is_crawler_running():
    time.sleep(refresh_rate)
    st.rerun()